In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
         continue
         # print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Milstone -1
Data Exploration & Baseline Submission
Perform EDA on the dataset (class distribution, audio length, gaps, ESC-50 noise).
Create a rule-based or random baseline submission on Kaggle.
Post questions about dataset, metrics (Macro F1), or unclear parts.

# Structure of data

In [ ]:
# List of Main Folder
import os

base_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

print("Main folders:")
print(os.listdir(base_path))

In [ ]:
# Content of Each Folder

for item in os.listdir(base_path):
    item_path = os.path.join(base_path, item)
    
    if os.path.isdir(item_path):
        print(f"\n** {item} contains:")
        print(os.listdir(item_path)[:12])  # only first 12 items


In [ ]:


# Number of genres of genres_stem

genres_path = os.path.join(base_path, "genres_stems")

genres = os.listdir(genres_path)

print("Genres:", genres)

In [ ]:


# First five audio files
import os
from IPython.display import Audio, display

# ESC-50 audio path
noise_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"

# first 5 files
audio_files = os.listdir(noise_path)[:5]

print("First 5 audio files:\n", audio_files)

# play them
for file in audio_files:
    file_path = os.path.join(noise_path, file)
    print("\nPlaying:", file)
    display(Audio(file_path))

In [ ]:


# content of each genre:

import os
from IPython.display import Audio, display

base_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

genres = os.listdir(base_path)

for genre in genres:
    print(f"\n🎧 Genre: {genre}")
    
    genre_path = os.path.join(base_path, genre)
    songs = os.listdir(genre_path)[:2]  # first 2 songs
    
    for song in songs:
        song_path = os.path.join(genre_path, song)
        
        # choose one instrument (vocals)
        audio_file = os.path.join(song_path, "vocals.wav")
        
        if os.path.exists(audio_file):
            print(f"▶ Playing: {song} - vocals.wav")
            display(Audio(audio_file))

In [ ]:
# Content of Mash
import os

mashup_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"

files = os.listdir(mashup_path)

print("Total mashups:", len(files))
print("First 5 files:", files[:5])

In [ ]:
# Playing of Content of Mashups
from IPython.display import Audio, display

for file in files[:3]:
    print("Playing:", file)
    display(Audio(os.path.join(mashup_path, file)))

# EDA[Expolatory Dta Analisis of Mashups]  

# Basic Properties of Audio
1.Duration   
2.Sample rate

3.Signal amplitude

In [ ]:
import os
import librosa
import numpy as np
import pandas as pd

base_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

data = []

for genre in os.listdir(base_path):
    genre_path = os.path.join(base_path, genre)
    songs = os.listdir(genre_path)[:2]   # 2 songs per genre
    
    for song in songs:
        audio_path = os.path.join(genre_path, song, "vocals.wav")
        
        try:
            y, sr = librosa.load(audio_path)
            
            duration = len(y) / sr
            min_amp = np.min(y)
            max_amp = np.max(y)
            
            data.append([genre, song, duration, sr, min_amp, max_amp])
        
        except:
            continue

# create dataframe
df = pd.DataFrame(data, columns=[
    "Genre", "Song", "Duration(sec)", "Sample Rate", "Min Amplitude", "Max Amplitude", 
])

# display
df.head(20)


# Waveform

In [ ]:
import os
import librosa
import matplotlib.pyplot as plt

base_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

genres = os.listdir(base_path)[0:10]  #  genres for clarity

for genre in genres:
    song = os.listdir(os.path.join(base_path, genre))[0]
    path = os.path.join(base_path, genre, song, "vocals.wav")
    
    y, sr = librosa.load(path)
    
    plt.figure(figsize=(15,3))
    plt.plot(y)
    plt.title(f"Waveform - {genre}")
    plt.show()

# Spectrogram


In [ ]:
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt

base_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

genres = os.listdir(base_path)[:10]  # compare 2 genres

for genre in genres:
    song = os.listdir(os.path.join(base_path, genre))[0]
    path = os.path.join(base_path, genre, song, "vocals.wav")
    
    y, sr = librosa.load(path)
    
    S = librosa.feature.melspectrogram(y=y, sr=sr)
    
    plt.figure(figsize=(10,4))
    librosa.display.specshow(librosa.power_to_db(S), sr=sr)
    plt.title(f"Spectrogram - {genre}")
    plt.colorbar()
    plt.show()

# Base line submission

In [ ]:
import pandas as pd

test_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"

test_df = pd.read_csv(test_path)

print(test_df.head())
print("Total samples:", len(test_df))

In [ ]:
genres = [
    "blues","classical","country","disco","hiphop",
    "jazz","metal","pop","reggae","rock"
]
import random

test_df["genre"] = [random.choice(genres) for _ in range(len(test_df))]
submission = test_df[["id", "genre"]]

submission.to_csv("submission.csv", index=False)

print(" submission.csv created!")

In [ ]:
submission.head(10)

# Milestone 2
clasical ML Baseline

Clean and preprocess audio stems.
Convert audio to numerical features (MFCCs, Spectrograms).
Train Logistic Regression, Naive Bayes, or boosting models (CatBoost, LightGBM, XGBoost).
Evaluate using Macro F1 and log results in W&B.

In [ ]:
try:
    import torch
    import torchaudio
    print("✅ Torch already installed")
except:
    print("Installing torch...")
    !pip install torch torchaudio -q

# imports

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# FEATURE FUNCTION (MFCC)

In [ ]:
def extract_features(file_path):
    y, sr = librosa.load(file_path, duration=30)
    
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    
    return mfcc_mean

# Create data set

In [ ]:
base_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

X = []
y = []

genres = os.listdir(base_path)

for genre in genres:
    genre_path = os.path.join(base_path, genre)
    songs = os.listdir(genre_path)
    
    for song in tqdm(songs[:50]):  # initially 50 (later full)
        file_path = os.path.join(genre_path, song, "vocals.wav")
        
        try:
            features = extract_features(file_path)
            X.append(features)
            y.append(genre)
        except:
            continue

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

# Train _Test Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Wandb init

In [ ]:
import wandb

wandb.init(
    project="23f2000441-t12026",
    name="Milestone-2-ML",
    mode="offline"   
)

# LOGISTIC REGRESSION

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)


model_lr = LogisticRegression(max_iter=300, solver='liblinear')


model_lr.fit(X_train_scaled, y_train)


y_pred = model_lr.predict(X_val_scaled)


f1 = f1_score(y_val, y_pred, average="macro")

print("Logistic Regression Macro F1:", f1)

# CATBOOST


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

model_cat = CatBoostClassifier(verbose=0)

model_cat.fit(X_train, y_train_enc)

pred_cat = model_cat.predict(X_val)

f1_cat = f1_score(y_val_enc, pred_cat, average="macro")

print("CatBoost F1:", f1_cat)

# Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import f1_score

model_nb = GaussianNB()
model_nb.fit(X_train, y_train)

pred_nb = model_nb.predict(X_val)

f1_nb = f1_score(y_val, pred_nb, average="macro")

print("Naive Bayes F1:", f1_nb)

# XGBOOST

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score


le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

model_xgb = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")

model_xgb.fit(X_train, y_train_enc)

pred_xgb = model_xgb.predict(X_val)

f1_xgb = f1_score(y_val_enc, pred_xgb, average="macro")

print("XGBoost F1:", f1_xgb)

# LIGHTGBM

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

model_lgb = LGBMClassifier()

model_lgb.fit(X_train, y_train_enc)

pred_lgb = model_lgb.predict(X_val)

f1_lgb = f1_score(y_val_enc, pred_lgb, average="macro")

print("LightGBM F1:", f1_lgb)

# COMPARISON

In [ ]:
results = {}

try:
    results["Logistic"] = f1
except:
    print("⚠ Logistic not found")


try:
    results["NaiveBayes"] = f1_nb
except:
    print("⚠ NaiveBayes not found")


try:
    results["XGBoost"] = f1_xgb
except:
    print("⚠ XGBoost not found")


try:
    results["LightGBM"] = f1_lgb
except:
    print("⚠ LightGBM not found")


try:
    results["CatBoost"] = f1_cat
except:
    print("⚠ CatBoost not found")


print("\n FINAL MODEL COMPARISON \n")

for model, score in results.items():
    print(f"{model}: {score:.4f}")

# Training for CatBoost

In [ ]:
import pandas as pd

test_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
test_df = pd.read_csv(test_path)

print(test_df.head())

In [ ]:
import librosa
import numpy as np

def extract_features(file_path):
    y, sr = librosa.load(file_path, duration=30)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    return np.mean(mfcc, axis=1)

In [ ]:
import os

mashup_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"

predictions = []

for file_id in test_df["id"]:
    
    
    file_name = f"song{int(file_id):04d}.wav"
    
    file_path = os.path.join(mashup_path, file_name)
    
    try:
        features = extract_features(file_path)
        features = features.reshape(1, -1)
        
        pred = model_cat.predict(features)
        
        genre = le.inverse_transform(pred)[0]
        
    except Exception as e:
        print("Error:", e)
        genre = "rock"
    
    predictions.append(genre)

test_df["genre"] = predictions

In [ ]:
submission = test_df[["id", "genre"]]

submission.to_csv("submission.csv", index=False)

print("✅ submission.csv created!")

In [ ]:
submission.head(10)

# Milestone-3
Your First Neural Network & CNNs!

Learn PyTorch basics: Tensors, Dataset (custom loader for training), DataLoader.
Convert audio to 2D/1D Mel-Spectrograms.
Build a simple CNN (Convolutional Neural Network)/NN (Neural Network) to process the spectrograms.
Implement training loop, loss, optimizer, and wandb logging.
Train and evaluate your CNN/NN (Neural Network)model.



In [ ]:
try:
    import torch
    import torchaudio
    print("✅ Torch already installed")
except:
    print("Installing torch...")
    !pip install torch torchaudio -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import librosa
import numpy as np
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

from torch.utils.data import TensorDataset, DataLoader



In [ ]:
import wandb

wandb.init(
    project="23f2000441-t12026",
    name="Milestone-3-CNN",
    mode="offline"   
)

# SPECTROGRAM FUNCTION

In [ ]:
def extract_spectrogram(file_path):
    y, sr = librosa.load(file_path, duration=30)
    
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    mel_db = mel_db[:128, :128]
    
    mean = np.mean(mel_db)
    std = np.std(mel_db)
    
    if std != 0:
        mel_db = (mel_db - mean) / std
    
    return mel_db

# Create Data Set

In [ ]:
base_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

X = []
y = []

stems = ["vocals.wav", "drums.wav", "bass.wav", "others.wav"]

for genre in os.listdir(base_path):
    genre_path = os.path.join(base_path, genre)
    
    for song in tqdm(os.listdir(genre_path)[:100]):
        
        for stem in stems:
            file_path = os.path.join(genre_path, song, stem)
            
            try:
                spec = extract_spectrogram(file_path)
                X.append(spec)
                y.append(genre)
            except:
                continue

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

# Label encoding

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y_enc, test_size=0.2, random_state=42
)


# Train -Test Split

In [ ]:
X_train = torch.tensor(X_train).unsqueeze(1).float()
X_val = torch.tensor(X_val).unsqueeze(1).float()

y_train = torch.tensor(y_train).long()
y_val = torch.tensor(y_val).long()

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)

# Torch data

# CNN Model


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 10)
        )
    
    def forward(self, x):
        return self.fc(self.conv(x))

# Training

In [ ]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

epochs = 30

for epoch in range(epochs):
    
    model.train()
    total_loss = 0
    
    for xb, yb in train_loader:
        
        outputs = model(xb)
        loss = criterion(outputs, yb)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss}")
    
    
    wandb.log({"loss": total_loss, "epoch": epoch+1})

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        preds = torch.argmax(outputs, axis=1)
        
        all_preds.extend(preds.numpy())
        all_labels.extend(yb.numpy())

f1 = f1_score(all_labels, all_preds, average="macro")

print(" CNN Macro F1:", f1)

wandb.log({"F1_score": f1})
wandb.finish()

# Milestone 4
Sequential Models & CRNNs (CNN + RNN/LSTM/GRU)

Learn about sequential models and their advantages for audio time-series.
Combine your CNN with an LSTM/GRU to create a CRNN (CNN extracts features, RNN captures the musical rhythm over time).
Train and compare with your pure CNN model.
Log and analyze results in W&B.

# Import

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import librosa
import numpy as np
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

from torch.utils.data import TensorDataset, DataLoader
import wandb

wandb.init(
    project="23f2000441-t12026",
    name="Milestone-4-CRNN",
    mode="offline"   # Kaggle ke liye best
)

# SPECTROGRAM FUNCTION

In [ ]:
def extract_spectrogram(file_path):
    y, sr = librosa.load(file_path, duration=5)  
    
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    mel_db = mel_db[:128, :128]
    
    mean = np.mean(mel_db)
    std = np.std(mel_db)
    
    if std != 0:
        mel_db = (mel_db - mean) / std
    
    return mel_db

# DATASET CREATE

In [ ]:
base_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

X = []
y = []

stems = ["vocals.wav", "drums.wav", "bass.wav", "others.wav"]

for genre in os.listdir(base_path):
    genre_path = os.path.join(base_path, genre)
    
    for song in tqdm(os.listdir(genre_path)[:100]):   # keep small first
        
        for stem in stems:
            file_path = os.path.join(genre_path, song, stem)
            
            try:
                spec = extract_spectrogram(file_path)
                X.append(spec)
                y.append(genre)
            except:
                continue

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

# LABEL ENCODE + SPLIT

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y_enc, test_size=0.2, random_state=42
)

# TORCH + DATALOADER

In [ ]:
X_train = torch.tensor(X_train).unsqueeze(1).float()
X_val = torch.tensor(X_val).unsqueeze(1).float()

y_train = torch.tensor(y_train).long()
y_val = torch.tensor(y_val).long()

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)

# CRNN Model

In [ ]:
class CRNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        # CNN part
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # LSTM part
        self.lstm = nn.LSTM(
            input_size=64 * 16,   # features
            hidden_size=128,
            batch_first=True
        )
        
        # FC
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.cnn(x)  
        

        x = x.permute(0, 3, 1, 2)  
        

        x = x.reshape(x.size(0), x.size(1), -1)  
        

        lstm_out, _ = self.lstm(x)

        out = lstm_out[:, -1, :] 

        out = self.fc(out)

        return out

# Training

In [ ]:
model = CRNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

epochs = 30

for epoch in range(epochs):
    
    model.train()
    total_loss = 0
    
    for xb, yb in train_loader:
        
        outputs = model(xb)
        loss = criterion(outputs, yb)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss}")
    
    
    wandb.log({
        "epoch": epoch+1,
        "loss": total_loss
    })

# Eval

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        preds = torch.argmax(outputs, axis=1)
        
        all_preds.extend(preds.numpy())
        all_labels.extend(yb.numpy())

f1 = f1_score(all_labels, all_preds, average="macro")

print("🔥 CRNN Macro F1:", f1)
wandb.log({"F1_score": f1})
wandb.finish()

# Milestone-5
Fine-Tuning Pre-trained Transformers

Learn transfer learning with Audio Transformers (AST/Hubert). Use Hugging Face feature extractors and audio models. Fine-tune on the noisy genre mashup dataset. Compare with your other models like CNN/CRNN results and log final performance.

In [ ]:
import os


import os

os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Completely silence absl + TF backend
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["KMP_WARNINGS"] = "0"

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module=r"torchaudio\..*")
warnings.filterwarnings("ignore", category=UserWarning, module=r"torio\..*")

# 0) Install + Imports

In [ ]:
!pip -q install torchaudio transformers accelerate evaluate

import os, random, math, gc
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

# 1) Label Map

In [ ]:
 ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
STEMS = os.path.join(ROOT, "genres_stems")
NOISE_DIR = os.path.join(ROOT, "random_noise")  # IF exists; else we'll search

GENRES = ["blues","classical","country","disco","hiphop","jazz","metal","pop","reggae","rock"]
label2id = {g:i for i,g in enumerate(GENRES)}
id2label = {i:g for g,i in label2id.items()}

# If you don't know noise folder name, find it:
if not os.path.exists(NOISE_DIR):
    for r, d, f in os.walk(ROOT):
        if "noise" in r.lower():
            NOISE_DIR = r
            break
print("NOISE_DIR:", NOISE_DIR)

# 2) Train

In [ ]:
def list_tracks():
    rows = []
    for g in GENRES:
        gp = os.path.join(STEMS, g)
        tracks = sorted([t for t in os.listdir(gp) if os.path.isdir(os.path.join(gp,t))])
        for t in tracks:
            tp = os.path.join(gp, t)
            row = {
                "genre": g,
                "label": label2id[g],
                "track_path": tp,
                "drums": os.path.join(tp, "drums.wav"),
                "bass": os.path.join(tp, "bass.wav"),
                "vocals": os.path.join(tp, "vocals.wav"),
                "other": os.path.join(tp, "other.wav"),
            }
            rows.append(row)
    return pd.DataFrame(rows)

train_df = list_tracks()
print(train_df.shape)
train_df.head()

# 3) Index Noise Files

In [ ]:
def list_noise_files(noise_dir):
    noise_files = []
    if os.path.exists(noise_dir):
        for r, d, f in os.walk(noise_dir):
            for fn in f:
                if fn.lower().endswith((".wav",".flac",".mp3",".ogg")):
                    noise_files.append(os.path.join(r, fn))
    return noise_files

noise_files = list_noise_files(NOISE_DIR)
print("Noise files:", len(noise_files))

# 4) Audio Utilities 

In [ ]:
TARGET_SR = 16000
CLIP_SECONDS = 10.0
CLIP_SAMPLES = int(TARGET_SR * CLIP_SECONDS)

def load_audio(path, sr=TARGET_SR):
    
    wav, s = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if s != sr:
        wav = torchaudio.functional.resample(wav, s, sr)
    return wav

def random_crop_or_pad(wav, n=CLIP_SAMPLES):
    L = wav.shape[1]
    if L == n:
        return wav
    if L > n:
        start = random.randint(0, L - n)
        return wav[:, start:start+n]
    # pad
    pad = n - L
    return F.pad(wav, (0, pad))

def rms(x):
    return torch.sqrt(torch.mean(x**2) + 1e-8)

def add_noise_at_snr(clean, noise, snr_db):
    clean_rms = rms(clean)
    noise_rms = rms(noise)
    
    target_noise_rms = clean_rms / (10**(snr_db/20))
    noise_scaled = noise * (target_noise_rms / (noise_rms + 1e-8))
    return clean + noise_scaled

# 5) (IMPORTANT) Synthetic Mashup Generator

In [ ]:


import random
import torch
import torchaudio

def rand_db(low=-6.0, high=6.0):
    return 10 ** (random.uniform(low, high) / 20.0)

def peak_norm(wav, eps=1e-8):
    # wav: (1, N)
    peak = wav.abs().max().clamp_min(eps)
    return wav / peak

def mix_stems(stems, gains=None):
    # stems: list of (1,N)
    if gains is None:
        gains = [1.0] * len(stems)
    mix = torch.zeros_like(stems[0])
    for s, g in zip(stems, gains):
        mix = mix + s * g
    # prevent clipping explosion
    mix = peak_norm(mix) * 0.9
    return mix

def add_noise_snr(clean, noise, snr_db):
    # clean/noise: (1,N)
    eps = 1e-8
    c_rms = torch.sqrt(torch.mean(clean**2) + eps)
    n_rms = torch.sqrt(torch.mean(noise**2) + eps)
    # scale noise to target SNR
    desired_n_rms = c_rms / (10 ** (snr_db / 20.0))
    noise_scaled = noise * (desired_n_rms / (n_rms + eps))
    out = clean + noise_scaled
    out = peak_norm(out) * 0.9
    return out

def make_one_mashup(genre_rows, noise_files, p_add_noise=0.8):
    """
    genre_rows: dataframe subset of one genre (contains paths to stems)
    Returns waveform (1, CLIP_SAMPLES)
    """
    # pick 4 different tracks from same genre
    picks = genre_rows.sample(4, replace=False).to_dict("records")

    # randomly choose one stem from each (drums/vocals/bass/other)
    # your index function should have columns: drums, vocals, bass, other
    drums = load_audio(picks[0]["drums"])
    vocals = load_audio(picks[1]["vocals"])
    bass  = load_audio(picks[2]["bass"])
    other = load_audio(picks[3]["other"])

    # random crop / pad to 10 sec
    drums  = random_crop_or_pad(drums)
    vocals = random_crop_or_pad(vocals)
    bass   = random_crop_or_pad(bass)
    other  = random_crop_or_pad(other)

    # random stem gains (realistic mixing)
    gains = [rand_db(-8, 2), rand_db(-10, 3), rand_db(-8, 2), rand_db(-6, 4)]
    wav = mix_stems([drums, vocals, bass, other], gains=gains)

    # optional random noise at random SNR
    if (len(noise_files) > 0) and (random.random() < p_add_noise):
        nf = random.choice(noise_files)
        nz = load_audio(nf)
        nz = random_crop_or_pad(nz)
        snr_db = random.uniform(0, 20)  # strong noise to mild
        wav = add_noise_snr(wav, nz, snr_db)

    return wav
    

# 6) Dataset Class (On-the-fly Mashups)


In [ ]:


from torch.utils.data import Dataset

class MashupTrainDataset(Dataset):
    def __init__(self, train_df, noise_files, n_samples=50000, seed=42):
        self.df = train_df.reset_index(drop=True)
        self.noise_files = noise_files
        self.n_samples = n_samples
        self.rng = np.random.RandomState(seed)

        
        self.by_genre = {g: self.df[self.df["genre"] == g] for g in GENRES}

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        g = GENRES[idx % len(GENRES)]  
        genre_rows = self.by_genre[g]

        wav = make_one_mashup(genre_rows, self.noise_files, p_add_noise=0.85)
        y = label2id[g]
        return wav.squeeze(0), y  

# 7) Model (AST / Audio Transformer)

In [ ]:


import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

model = AutoModelForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(GENRES),
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.data.clone()

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.data, alpha=1 - self.decay)

    @torch.no_grad()
    def apply_to(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.data.clone()
                p.data.copy_(self.shadow[n])

    @torch.no_grad()
    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.backup[n])
        self.backup = {}

ema = EMA(model, decay=0.999)
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module=r"torchaudio\..*")
warnings.filterwarnings("ignore", category=UserWarning, module=r"torio\..*")

# 8) Collate Function

In [ ]:


import torch

def spec_augment(mels, time_mask=40, freq_mask=16, p=0.8):
    
    if random.random() > p:
        return mels
    B, T, F = mels.shape

    
    t = random.randint(0, time_mask)
    t0 = random.randint(0, max(0, T - t))
    mels[:, t0:t0+t, :] = 0.0

    
    f = random.randint(0, freq_mask)
    f0 = random.randint(0, max(0, F - f))
    mels[:, :, f0:f0+f] = 0.0

    return mels

def collate_fn(batch):
    wavs, ys = zip(*batch)
    wavs = [w.numpy() for w in wavs]  
    inputs = feature_extractor(
        wavs,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        padding=True
    )
    
    if "input_values" in inputs:
        inputs["input_values"] = spec_augment(inputs["input_values"])
    y = torch.tensor(ys, dtype=torch.long)
    return inputs, y

# 9) Train Loop

In [ ]:


import torch
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

def label_smooth_ce(logits, y, eps=0.1):
    
    num_classes = logits.size(-1)
    log_probs = torch.log_softmax(logits, dim=-1)
    nll = -log_probs.gather(dim=-1, index=y.unsqueeze(1)).squeeze(1)
    smooth = -log_probs.mean(dim=-1)
    loss = (1 - eps) * nll + eps * smooth
    return loss.mean()

@torch.no_grad()
def evaluate_probe(model, probe_df, n_crops=3):
    model.eval()
    y_true, y_pred = [], []

    for _, r in probe_df.iterrows():
        
        
        wav = load_audio(r["other"])
        logits_sum = None
        for _ in range(n_crops):
            clip = random_crop_or_pad(wav).squeeze(0).numpy()
            inp = feature_extractor([clip], sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
            for k in inp: inp[k] = inp[k].to(device)
            lg = model(**inp).logits
            logits_sum = lg if logits_sum is None else (logits_sum + lg)
        pred = int(torch.argmax(logits_sum / n_crops, dim=1).cpu().item())
        y_true.append(label2id[r["genre"]])
        y_pred.append(pred)

    return f1_score(y_true, y_pred, average="macro")

scaler = torch.amp.GradScaler("cuda")

def train_one_epoch(model, loader, optim, ema=None, smooth_eps=0.1):
    model.train()
    total = 0.0

    for inputs, y in tqdm(loader, desc="train", leave=False):
        for k in inputs:
            inputs[k] = inputs[k].to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optim.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda"):
            out = model(**inputs)
            loss = label_smooth_ce(out.logits, y, eps=smooth_eps)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optim)
        scaler.update()

        if ema is not None:
            ema.update(model)

        total += loss.item()

    return total / max(1, len(loader))

In [ ]:
import torch, os
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 10) Training

In [ ]:



import os, gc, math, numpy as np, torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

if "NOISE_FILES" not in globals():
    if "noise_files" in globals():
        NOISE_FILES = noise_files
    else:
        raise NameError("noise_files / NOISE_FILES not found in Step0-9")

required = ["train_df","MashupTrainDataset","collate_fn","model","device",
            "feature_extractor","TARGET_SR","label2id","id2label","ema",
            "load_audio","random_crop_or_pad"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing from Step0-9: {missing}")

CKPT_DIR = "/kaggle/working"
os.makedirs(CKPT_DIR, exist_ok=True)

torch.backends.cudnn.benchmark = True
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

# -------- Light SpecAug (keep mild so training faster/stable) --------
def spec_augment(x, time_mask=40, freq_mask=8, p=0.5):
    if torch.rand(1).item() > p:
        return x
    if x.dim() == 3:
        x = x.unsqueeze(1)  # [B,1,T,F]
    B,C,T,F = x.shape

    f = int(torch.randint(0, freq_mask+1, (1,)).item())
    if f > 0:
        f0 = int(torch.randint(0, max(1, F-f), (1,)).item())
        x[:, :, :, f0:f0+f] = 0

    t = int(torch.randint(0, time_mask+1, (1,)).item())
    if t > 0:
        t0 = int(torch.randint(0, max(1, T-t), (1,)).item())
        x[:, :, t0:t0+t, :] = 0

    return x.squeeze(1)


def mixup_batch(inputs, y, alpha=0.3):
    if alpha <= 0:
        return inputs, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(y.size(0), device=y.device)
    inputs2 = {k: v[idx] for k, v in inputs.items()}
    mixed = {k: lam*inputs[k] + (1-lam)*inputs2[k] for k in inputs}
    return mixed, y, y[idx], lam

def mixup_ce(logits, y_a, y_b, lam, label_smoothing=0.1):
    C = logits.size(-1)
    logp = torch.log_softmax(logits, dim=-1)
    ya = torch.zeros((y_a.size(0), C), device=logits.device).scatter_(1, y_a[:,None], 1.0)
    yb = torch.zeros((y_b.size(0), C), device=logits.device).scatter_(1, y_b[:,None], 1.0)
    ya = (1-label_smoothing)*ya + label_smoothing/C
    yb = (1-label_smoothing)*yb + label_smoothing/C
    target = lam*ya + (1-lam)*yb
    return -(target*logp).sum(dim=1).mean()

# -------- quick probe (fast) --------
val_probe = train_df.groupby("genre").head(15).reset_index(drop=True)

@torch.no_grad()
def probe_f1(n_crops=2):
    model.eval()
    y_true, y_pred = [], []
    from sklearn.metrics import f1_score
    for _, r in val_probe.iterrows():
        wav = load_audio(r["other"])
        lg_sum = None
        for _ in range(n_crops):
            clip = random_crop_or_pad(wav).squeeze(0).numpy()
            inp = feature_extractor([clip], sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
            for k in inp: inp[k] = inp[k].to(device)
            lg = model(**inp).logits
            lg_sum = lg if lg_sum is None else (lg_sum + lg)
        pred = int(torch.argmax(lg_sum, dim=1).cpu().item())
        y_true.append(label2id[r["genre"]])
        y_pred.append(pred)
    return f1_score(y_true, y_pred, average="macro")

@torch.no_grad()
def probe_f1_with_ema():
    ema.apply_to(model)
    f1 = probe_f1(n_crops=2)
    ema.restore(model)
    return f1

SCHEDULE = [
    {"n": 50000,  "bs": 8,  "lr": 2e-5},
    {"n": 80000,  "bs": 10, "lr": 2e-5},
    {"n": 100000, "bs": 10, "lr": 1.5e-5},
]

optim = torch.optim.AdamW(model.parameters(), lr=SCHEDULE[0]["lr"], weight_decay=0.01)
total_steps = sum(math.ceil(s["n"]/s["bs"]) for s in SCHEDULE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=total_steps)

best_f1 = -1.0
best_path = os.path.join(CKPT_DIR, "best_model_ema.pth")

for ep, cfg in enumerate(SCHEDULE, start=1):
    for pg in optim.param_groups: pg["lr"] = cfg["lr"]

    train_ds = MashupTrainDataset(train_df, NOISE_FILES, n_samples=cfg["n"], seed=1234+ep)
    train_loader = DataLoader(train_ds, batch_size=cfg["bs"], shuffle=True,
                              num_workers=2, pin_memory=True, collate_fn=collate_fn)

    model.train()
    total_loss = 0.0

    for inputs, y in tqdm(train_loader, desc=f"epoch{ep}", leave=False):
        for k in inputs: inputs[k] = inputs[k].to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        # mild specaug
        if "input_values" in inputs:
            inputs["input_values"] = spec_augment(inputs["input_values"])

        optim.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            mixed, ya, yb, lam = mixup_batch(inputs, y, alpha=0.3)
            logits = model(**mixed).logits
            loss = mixup_ce(logits, ya, yb, lam, label_smoothing=0.1)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optim)
        scaler.update()

        ema.update(model)
        scheduler.step()
        total_loss += float(loss.item())

    avg_loss = total_loss / max(1, len(train_loader))
    f1 = probe_f1_with_ema()

    print(f"Epoch {ep} | n={cfg['n']} bs={cfg['bs']} lr={cfg['lr']:.1e} | loss={avg_loss:.4f} | probe_f1={f1:.4f}")

    # save best EMA
    if f1 > best_f1:
        best_f1 = f1
        ema.apply_to(model)
        torch.save(model.state_dict(), best_path)
        ema.restore(model)
        print("✅ Updated best_model_ema.pth | best_f1 =", best_f1)

    del train_ds, train_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("✅ Training done. Best:", best_path)

# 11) Test Inference + Submission

In [ ]:


import os, gc
import pandas as pd
import torch
from tqdm.auto import tqdm

CKPT_DIR = "/kaggle/working"
best_path = os.path.join(CKPT_DIR, "best_model_ema.pth")
assert os.path.exists(best_path), "best_model_ema.pth not found. Run Step10 first."

model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

TEST_CSV = os.path.join(ROOT, "test.csv")
SUB_CSV  = os.path.join(ROOT, "sample_submission.csv")
test_df = pd.read_csv(TEST_CSV)
sub = pd.read_csv(SUB_CSV)

file_col = "filename"
N_CROPS = 15  # strong but not too slow

@torch.no_grad()
def predict_one(path, n_crops=15):
    wav_full = load_audio(path)
    lg_sum = None
    for _ in range(n_crops):
        clip = random_crop_or_pad(wav_full).squeeze(0).numpy()
        inp = feature_extractor([clip], sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
        for k in inp: inp[k] = inp[k].to(device)
        lg = model(**inp).logits
        lg_sum = lg if lg_sum is None else (lg_sum + lg)
    pred = int(torch.argmax(lg_sum, dim=1).cpu().item())
    return pred

pred_ids = []
for rel in tqdm(test_df[file_col].astype(str).tolist(), desc="Predict"):
    fpath = os.path.join(ROOT, rel)   # ROOT + "mashups/song0001.wav"
    pred_ids.append(predict_one(fpath, n_crops=N_CROPS))

sub["genre"] = [id2label[i] for i in pred_ids]
out_name = f"submission_bestEMA_tta{N_CROPS}.csv"
sub.to_csv(out_name, index=False)

print("Saved:", out_name)
print(sub.head())